In [1]:
import pandas as pd
import geopandas as gpd
import os
import math
import numpy as np

os.environ['SHAPE_RESTORE_SHX'] = 'YES'

In [2]:
df = pd.read_csv("C:/Users/Century-PC/Desktop/GAM_assurance/buildings (1).csv")

In [3]:
df.head()

,NUMERO_POLICE,wilaya_id,WILAYA,COMMUNE,zone_sismique,zone_ord,type_batiment,building_class,sum_insured,prime_nette,...,viol_etages,viol_ratio_plan,viol_densite_murs,viol_epaisseur,viol_mortier,viol_beton,viol_dist_murs,tax_rate,damage_ratio,expected_payout
0,1.633000e+13,9,BLIDA,BIRTOUTA,III,4,Industriel,3A,1000000,2500.0,...,0,0,1,0,0,0,1,0.30,0.4470,447000
1,3.430300e+13,34,B.B ARRERIDJ,BORDJ BOU ARRERIDJ,IIa,2,Industriel,3B,1000000,2500.0,...,0,0,0,0,1,0,1,0.14,0.1950,195000
2,1.930600e+13,19,SETIF,EL EULMA,IIa,2,Industriel,3B,60000000,25800.0,...,0,1,0,0,1,0,0,0.14,0.4467,26802000
3,2.330400e+13,23,ANNABA,AIN BERDA,IIa,2,Industriel,3B,50000000,21500.0,...,0,0,1,0,0,0,0,0.14,0.0885,4425000
4,6.309000e+12,6,BEJAIA,TAZMALT,IIa,2,Industriel,3A,2000000,2500.0,...,0,0,0,0,0,0,0,0.12,0.1095,219000


In [ ]:

df["zone_sismique"]  = df["zone_sismique"].astype(str).str.strip()
df["building_class"] = df["building_class"].astype(str).str.strip()

# Derived features :
df["age_batiment"]       = 2025 - df["age_construction"]
df["ratio_longlarg"]     = df["longueur"]  / df["largeur"]
df["densite_murs"]       = df["aire_murs"] / df["surface_plancher"]
df["ratio_hauteur_larg"] = df["hauteur"]   / df["largeur"]


#  Tax matrix : 
tax_matrix = {
    ("1A","0"):0.05, ("1A","I"):0.15, ("1A","IIa"):0.25, ("1A","IIb"):0.30, ("1A","III"):0.40,
    ("1B","0"):0.06, ("1B","I"):0.12, ("1B","IIa"):0.20, ("1B","IIb"):0.25, ("1B","III"):0.30,
    ("2A","0"):0.04, ("2A","I"):0.10, ("2A","IIa"):0.15, ("2A","IIb"):0.20, ("2A","III"):0.25,
    ("2B","0"):0.05, ("2B","I"):0.08, ("2B","IIa"):0.12, ("2B","IIb"):0.17, ("2B","III"):0.22,
    ("3A","0"):0.03, ("3A","I"):0.07, ("3A","IIa"):0.10, ("3A","IIb"):0.14, ("3A","III"):0.18,
    ("3B","0"):0.04, ("3B","I"):0.06, ("3B","IIa"):0.9, ("3B","IIb"):0.12, ("3B","III"):0.16,
}
df["tax_rate"]        = df.apply(lambda r: tax_matrix.get((r["building_class"], r["zone_sismique"]), np.nan), axis=1)
df["expected_payout"] = df["tax_rate"] * df["sum_insured"]
df["zone_sismique"] = df["zone_sismique"].str.replace("IIA", "IIa").str.replace("IIB", "IIb")
#  Zone -> ordinal :
zone_ord = {"0": 0, "I": 1, "IIa": 2, "IIb": 3, "III": 4}
df["zone_ord"] = df["zone_sismique"].map(zone_ord)
df["building_class"] = df["building_class"].astype(str).str.strip()
df["zone_sismique"] = df["zone_sismique"].astype(str).str.strip()

#  Handle missing values :
num_cols = ["age_batiment", "ratio_longlarg", "densite_murs",
            "resistance_mortier", "resistance_beton", "distance_entre_murs"]
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

df["damage_ratio"] = df["damage_ratio"].clip(0, 1)

#  Train / score split :

df_train = df.dropna(subset=["damage_ratio"]).copy()
df_score = df[df["damage_ratio"].isna()].copy()

print(f"Training rows : {len(df_train):,}")
print(f"Rows to score : {len(df_score):,}")

Training rows : 37,104
Rows to score : 0


In [5]:
df_train[["zone_sismique","building_class","tax_rate"]].head(-20)

,zone_sismique,building_class,tax_rate
0,III,3A,0.18
1,IIa,3B,0.90
2,IIa,3B,0.90
3,IIa,3B,0.90
4,IIa,3A,0.10
...,...,...,...
37079,0,1A,0.05
37080,IIa,1A,0.25
37081,IIa,1A,0.25
37082,III,1B,0.30


In [ ]:
FEATURES = [
    # Seismic context
    "zone_ord",
    # Geometry
    "nb_niveaux", "hauteur", "ratio_longlarg",
    "ratio_hauteur_larg", "densite_murs", "distance_entre_murs",
    # Materials
    "epaisseur_mur", "resistance_mortier", "resistance_beton",
    # Age
    "age_batiment",
    # RPA compliance signals  ← the key bridge between your function and XGBoost
    "rpa_conforme", "rpa_nb_violations",
    "viol_hauteur", "viol_etages", "viol_ratio_plan",
    "viol_densite_murs", "viol_epaisseur", "viol_mortier",
    "viol_beton", "viol_dist_murs",
    # Insurance 
    "tax_rate",           
]

TARGET = "damage_ratio"

In [7]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [8]:
X = df_train[FEATURES]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
my_model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,       
    reg_lambda=1.0,     
    objective="reg:squarederror",
    random_state=42
)

In [10]:
my_model.fit(X_train,y_train,
            eval_set=[(X_test, y_test)],
          verbose=50)

[0]	validation_0-rmse:0.24805
[50]	validation_0-rmse:0.21269
[100]	validation_0-rmse:0.21218
[150]	validation_0-rmse:0.21225
[200]	validation_0-rmse:0.21251
[250]	validation_0-rmse:0.21259
[300]	validation_0-rmse:0.21280
[350]	validation_0-rmse:0.21303
[399]	validation_0-rmse:0.21326


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [11]:
from sklearn.metrics import mean_absolute_error

In [12]:
df_score = df.head(10).copy()

In [13]:
df_score

,NUMERO_POLICE,wilaya_id,WILAYA,COMMUNE,zone_sismique,zone_ord,type_batiment,building_class,sum_insured,prime_nette,...,viol_etages,viol_ratio_plan,viol_densite_murs,viol_epaisseur,viol_mortier,viol_beton,viol_dist_murs,tax_rate,damage_ratio,expected_payout
0,1.633000e+13,9,BLIDA,BIRTOUTA,III,4,Industriel,3A,1000000,2500.0,...,0,0,1,0,0,0,1,0.18,0.4470,180000.0
1,3.430300e+13,34,B.B ARRERIDJ,BORDJ BOU ARRERIDJ,IIa,2,Industriel,3B,1000000,2500.0,...,0,0,0,0,1,0,1,0.90,0.1950,900000.0
2,1.930600e+13,19,SETIF,EL EULMA,IIa,2,Industriel,3B,60000000,25800.0,...,0,1,0,0,1,0,0,0.90,0.4467,54000000.0
3,2.330400e+13,23,ANNABA,AIN BERDA,IIa,2,Industriel,3B,50000000,21500.0,...,0,0,1,0,0,0,0,0.90,0.0885,45000000.0
4,6.309000e+12,6,BEJAIA,TAZMALT,IIa,2,Industriel,3A,2000000,2500.0,...,0,0,0,0,0,0,0,0.10,0.1095,200000.0
5,4.230300e+13,42,TIPAZA,BOU ISMAIL,III,4,Industriel,3A,6000000,3000.0,...,0,0,1,0,0,0,0,0.18,0.2798,1080000.0
6,1.530900e+13,15,TIZI OUZOU,TIGZIRT,IIa,2,Industriel,3A,44000000,18920.0,...,0,0,0,0,0,1,0,0.10,0.3045,4400000.0
7,1.634000e+13,35,BOUMERDES,KHEMIS EL KHECHNA,III,4,Industriel,3B,11000000,5170.0,...,0,0,1,0,0,0,0,0.16,0.5178,1760000.0
8,3.430300e+13,34,B.B ARRERIDJ,BORDJ BOU ARRERIDJ,IIa,2,Industriel,3B,1000000,2500.0,...,0,0,0,1,1,0,0,0.90,0.5455,900000.0
9,9.306000e+12,9,BLIDA,SOUMAA BLIDA,III,4,Industriel,3B,1000000,2500.0,...,0,0,0,0,1,0,0,0.16,0.6964,160000.0


In [ ]:

df_score["age_batiment"] = 2025 - df_score["age_construction"]
df_score["ratio_longlarg"] = df_score["longueur"] / df_score["largeur"]
df_score["densite_murs"] = df_score["aire_murs"] / df_score["surface_plancher"]
df_score["ratio_hauteur_larg"] = df_score["hauteur"] / df_score["largeur"]
df_score["zone_ord"] = df_score["zone_sismique"].map({"0": 0, "I": 1, "IIa": 2, "IIb": 3, "III": 4})

#testing snippet
df_score["predicted_damage_ratio"] = my_model.predict(df_score[FEATURES]).clip(0, 1)

df_score["predicted_payout"] = df_score["predicted_damage_ratio"] * df_score["sum_insured"]

# Simple risk-loaded premium
LOADING_FACTOR = 1.25  
df_score["suggested_premium"] = df_score["predicted_payout"] * LOADING_FACTOR

# Flag where XGBoost disagrees strongly with the tax matrix
df_score["delta_vs_tax"] = df_score["predicted_damage_ratio"] - df_score["tax_rate"]
conditions = [
    (df_score["delta_vs_tax"].abs() > 0.20),          # Condition 1
    (df_score["delta_vs_tax"].abs() > 0.15)           # Condition 2
]

choices = ["red", "orange"]
df_score["flag_review"] = np.select(conditions, choices, default="green")


df_score[["NUMERO_POLICE", "predicted_damage_ratio", "tax_rate", "suggested_premium", "flag_review"]]


,NUMERO_POLICE,predicted_damage_ratio,tax_rate,suggested_premium,flag_review
0,1.633000e+13,0.524520,0.18,6.556502e+05,red
1,3.430300e+13,0.285156,0.90,3.564451e+05,red
2,1.930600e+13,0.449218,0.90,3.369133e+07,red
3,2.330400e+13,0.244691,0.90,1.529320e+07,red
4,6.309000e+12,0.340384,0.10,8.509593e+05,red
5,4.230300e+13,0.436453,0.18,3.273396e+06,red
6,1.530900e+13,0.529055,0.10,2.909804e+07,red
7,1.634000e+13,0.547371,0.16,7.526354e+06,red
8,3.430300e+13,0.501909,0.90,6.273866e+05,red
9,9.306000e+12,0.608298,0.16,7.603725e+05,red


In [15]:
df_perc = df_score

In [16]:
df_perc[["NUMERO_POLICE", "predicted_damage_ratio", "tax_rate", "suggested_premium", "flag_review"]].tail()

,NUMERO_POLICE,predicted_damage_ratio,tax_rate,suggested_premium,flag_review
5,4.230300e+13,0.436453,0.18,3.273396e+06,red
6,1.530900e+13,0.529055,0.10,2.909804e+07,red
7,1.634000e+13,0.547371,0.16,7.526354e+06,red
8,3.430300e+13,0.501909,0.90,6.273866e+05,red
9,9.306000e+12,0.608298,0.16,7.603725e+05,red


In [17]:
df_perc.head(2)

,NUMERO_POLICE,wilaya_id,WILAYA,COMMUNE,zone_sismique,zone_ord,type_batiment,building_class,sum_insured,prime_nette,...,viol_beton,viol_dist_murs,tax_rate,damage_ratio,expected_payout,predicted_damage_ratio,predicted_payout,suggested_premium,delta_vs_tax,flag_review
0,1.633000e+13,9,BLIDA,BIRTOUTA,III,4,Industriel,3A,1000000,2500.0,...,0,1,0.18,0.447,180000.0,0.524520,524520.158768,655650.19846,0.344520,red
1,3.430300e+13,34,B.B ARRERIDJ,BORDJ BOU ARRERIDJ,IIa,2,Industriel,3B,1000000,2500.0,...,0,1,0.90,0.195,900000.0,0.285156,285156.041384,356445.05173,-0.614844,red


In [18]:
import pickle

In [21]:
filename = "risk_assessment.pkl"

In [23]:
with open(filename, "wb") as file:
    pickle.dump(my_model, file)

print(f"Model successfully saved as {filename}")

Model successfully saved as risk_assessment.pkl
